In [ ]:
import os
import sys
notebook_dir = os.getcwd()
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))  # Add the project root directory to the path
from dataclasses import dataclass
import warnings
from math import floor, ceil
import h5py
import numpy as np
import pandas as pd
import env_reader
import metadata_reader as mr
from custom_io import open_file, get_datetime_for_fname
import matplotlib.pyplot as plt

In [ ]:
fpath_tmev_data = open_file("Choose TMEV dataset (assembled traces)")
fpath_stim_data = open_file("Choose stim dataset (assembled traces)")
fpath_delays_amplitudes = open_file("Choose SD delays and amplitudes dataset (excel file)")

In [ ]:
df_delays_amplitudes = pd.read_excel(fpath_delays_amplitudes)

In [ ]:
df_delays_amplitudes

In [ ]:
env_dict = env_reader.read_env()
mdata = mr.MetadataReader.from_env_dict(env_dict)

In [ ]:
dict_euuid_trace = {}
with h5py.File(fpath_tmev_data, 'r') as f:
    for i_row, row in df_delays_amplitudes[df_delays_amplitudes["exp_type"] == "tmev"].iterrows():
        event_uuid = row["event_uuid"]
        trace = f[event_uuid]["mean_fluo"][()]
        dict_euuid_trace[event_uuid] = trace

dict_stim_begin_end_frames = {}
with h5py.File(fpath_stim_data, 'r') as f:
    for i_row, row in df_delays_amplitudes[df_delays_amplitudes["exp_type"] != "tmev"].iterrows():
        event_uuid = row["event_uuid"]
        trace = f[event_uuid]["mean_fluo"][()]
        i_stim_begin = f[event_uuid].attrs["i_stim_begin_frame"]
        i_stim_end = f[event_uuid].attrs["i_stim_end_frame"]
        dict_stim_begin_end_frames[event_uuid] = (i_stim_begin, i_stim_end)
        trace[i_stim_begin-15:i_stim_end] = trace[i_stim_begin-16]*0.7  # Set the stim period to less outstanding value for plotting
        dict_euuid_trace[event_uuid] = trace



In [ ]:
fig, axs = plt.subplots(len(dict_euuid_trace), 1, figsize=(10, 6*len(dict_euuid_trace)), sharex=False)
for i_row, (event_uuid, trace) in enumerate(dict_euuid_trace.items()):
    i_bl = df_delays_amplitudes[df_delays_amplitudes["event_uuid"] == event_uuid].i_bl.iloc[0]
    i_sd1 = df_delays_amplitudes[df_delays_amplitudes["event_uuid"] == event_uuid].i_sd1.iloc[0]
    i_sd2 = df_delays_amplitudes[df_delays_amplitudes["event_uuid"] == event_uuid].i_sd2.iloc[0]
    print(f"{event_uuid}: {i_sd1 - i_plot_begin}, {i_sd2 - i_plot_begin}")
    i_plot_begin = i_bl-10
    i_plot_end = i_sd2+150
    axs[i_row].plot(trace[i_plot_begin:i_plot_end])
    # plot sd windows
    # plot sd maxima
    axs[i_row].vlines([i_sd1 - i_plot_begin, i_sd2 - i_plot_begin], 0, 1.5*trace.max(), color='r', linestyle='--', label="Baseline")
    axs[i_row].set_title(event_uuid)
    axs[i_row].set_ylabel("Fluorescence (a.u.)")
    axs[i_row].set_xlabel("Time (s)")
    axs[i_row].grid(True)
plt.tight_layout()
plt.show()